# v1.3 / 7 标准卷积检测器对照与框尺度优化

独立验证表明MobileNet候选没有达到可用精度；已有五层CNN的定位更好。本实验仍严格导出标准全INT8 TFLite，并将膨胀卷积等价展开为普通5×5卷积，避免依赖特殊加速路径。不是恢复自定义HEX推理。小手框的宽高损失改为相对误差，缓解大、小手框在归一化坐标上的权重失衡。

In [1]:
# v1.3: project-only Jupyter execution; no board programming.
import os,sys,json,hashlib
from pathlib import Path
import numpy as np
import tensorflow as tf
ROOT=Path.cwd()
while not (ROOT/'AGENTS.md').exists():
    if ROOT==ROOT.parent: raise RuntimeError('Open this notebook inside the CNN project')
    ROOT=ROOT.parent
assert ROOT.name=='fpga-w.-codex-cnn'
sys.path.insert(0,str(ROOT/'ml/tinyml'))
from workflow import export_int8,predict_tflite,digest
os.environ['KERAS_HOME']=str(ROOT/'ml/tinyml/keras')
tf.keras.utils.set_random_seed(1301)
tf.config.threading.set_intra_op_parallelism_threads(6)
tf.config.threading.set_inter_op_parallelism_threads(2)
print('TensorFlow',tf.__version__,'GPU',tf.config.list_physical_devices('GPU'))


TensorFlow 2.16.2 GPU []


In [2]:
sys.path.insert(0,str(ROOT/'ml/detector'))
from train import data,encode,augment
from export import stats
from workflow import split_detection_heads
training=data('train');validation=data('val')
out=ROOT/'ml/tinyml/artifacts/detector_conv';out.mkdir(parents=True,exist_ok=True)
original=tf.keras.models.load_model(ROOT/'ml/detector/deeper256/best.keras',compile=False)
inputs=tf.keras.Input((144,256,1),name='gray');x=inputs
for source in original.layers[1:]:
    config=source.get_config()
    if source.name=='conv3':config['kernel_size']=(5,5);config['dilation_rate']=(1,1)
    layer=source.__class__.from_config(config);x=layer(x)
    weights=source.get_weights()
    if source.name=='conv3':
        w=weights[0];expanded=np.zeros((5,5,*w.shape[2:]),np.float32);expanded[::2,::2]=w;weights=[expanded]
    layer.set_weights(weights)
model=tf.keras.Model(inputs,x,name='hand_standard_conv_v1_3')
probe=np.stack([r[0] for r in validation[:8]])[...,None].astype(np.float32)
error=float(np.max(np.abs(original.predict((probe-128)/128,verbose=0)-model.predict((probe-128)/128,verbose=0))))
assert error<1e-4
print('Equivalent convolution max absolute error',error)
@tf.keras.utils.register_keras_serializable(package='gesture_v1_3')
def relative_box_loss(y,p):
    target=y[...,0];prob=tf.clip_by_value(tf.sigmoid(p[...,0]),1e-5,1-1e-5)
    pos=tf.cast(target>=.999,tf.float32);denom=tf.maximum(tf.reduce_sum(pos,axis=(1,2)),1.)
    focal=-pos*(1-prob)**2*tf.math.log(prob)-(1-pos)*(1-target)**4*prob**2*tf.math.log(1-prob)
    scales=tf.concat([tf.ones_like(y[...,1:3]),tf.maximum(y[...,3:5],.02)],axis=-1)
    d=tf.abs((p[...,1:5]-y[...,1:5])/scales)
    smooth=tf.where(d<.1,5*d*d,d-.05)
    regression=tf.reduce_sum(smooth*tf.constant([1.,1.,.5,.5]),axis=-1)*y[...,5]
    return tf.reduce_mean(tf.reduce_sum(focal+2*regression,axis=(1,2))/denom)
model.compile(tf.keras.optimizers.Adam(.0002),loss=relative_box_loss)
model.summary()


Equivalent convolution max absolute error 0.0


Model: "hand_standard_conv_v1_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gray (InputLayer)               │ (None, 144, 256, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv0 (Conv2D)                  │ (None, 72, 128, 4)     │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn0 (BatchNormalization)        │ (None, 72, 128, 4)     │            16 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu0 (ReLU)                    │ (None, 72, 128, 4)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 36, 64, 8)      │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 36, 64, 8)      │            32 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu1 (ReLU)                    │ (None, 36, 64, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 18, 32, 16)     │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 18, 32, 16)     │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu2 (ReLU)                    │ (None, 18, 32, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 18, 32, 32)     │        12,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3 (BatchNormalization)        │ (None, 18, 32, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu3 (ReLU)                    │ (None, 18, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv4 (Conv2D)                  │ (None, 18, 32, 32)     │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn4 (BatchNormalization)        │ (None, 18, 32, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu4 (ReLU)                    │ (None, 18, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_deep (Conv2D)              │ (None, 18, 32, 5)      │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,025 (93.85 KB)

 Trainable params: 23,841 (93.13 KB)

 Non-trainable params: 184 (736.00 B)

In [3]:
rng=np.random.default_rng(1301)
def generator():
    while True:
        for i in rng.permutation(len(training)):
            im,b,_=training[i];im,b=augment(im,b,rng)
            yield ((im.astype(np.float32)-128)/128)[...,None],encode(b)
options=tf.data.Options();options.threading.private_threadpool_size=4
ds=tf.data.Dataset.from_generator(generator,output_signature=(tf.TensorSpec((144,256,1),tf.float32),tf.TensorSpec((18,32,6),tf.float32))).batch(48).prefetch(2).with_options(options)
vx=np.stack([r[0] for r in validation])[...,None].astype(np.float32)
vy=np.stack([encode(r[1]) for r in validation])
vd=tf.data.Dataset.from_tensor_slices(((vx-128)/128,vy)).batch(48).with_options(options)
history=model.fit(ds,steps_per_epoch=(len(training)+47)//48,validation_data=vd,epochs=12,verbose=2,callbacks=[tf.keras.callbacks.ModelCheckpoint(str(out/'selected.keras'),monitor='val_loss',save_best_only=True),tf.keras.callbacks.CSVLogger(str(out/'training.csv'))])
model=tf.keras.models.load_model(out/'selected.keras',compile=False)


Epoch 1/12


C:\Users\francis\Desktop\fpga-w.-codex-cnn\ml\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


111/111 - 13s - 115ms/step - loss: 1.3915 - val_loss: 1.4139


Epoch 2/12


111/111 - 9s - 77ms/step - loss: 1.3962 - val_loss: 1.3533


...

111/111 - 9s - 78ms/step - loss: 1.3791 - val_loss: 1.3533


Epoch 11/12


111/111 - 9s - 79ms/step - loss: 1.3568 - val_loss: 1.3170


Epoch 12/12


111/111 - 9s - 81ms/step - loss: 1.3672 - val_loss: 1.3489


## 独立INT8检测头
分离object logit和坐标的输出量化尺度；两头浮点输出应与训练时的5通道输出完全一致。原先固定Q8的RTL不能直接执行这里的TFLite模型。

In [4]:
deployment=split_detection_heads(model)
p=model.predict((vx[:8]-128)/128,verbose=0)
a,b=deployment.predict((vx[:8]-128)/128,verbose=0)
assert np.max(np.abs(np.concatenate([a,b],axis=-1)-p))<1e-5
indices=np.random.default_rng(1301).choice(len(training),480,replace=False)
calibration=(np.stack([training[i][0] for i in indices])[...,None].astype(np.float32)-128)/128
path,contract=export_int8(deployment,calibration,out,'hand_detector_int8')
print(json.dumps(contract,indent=2))


C:\Users\francis\Desktop\fpga-w.-codex-cnn\ml\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:964: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


{
  "version": "v1.3",
  "sha256": "27e969f1614a9875d6d95ebfb4664422ba8de80f6c1f44afec0ea33938d21019",
  "bytes": 30616,
  "inputs": [
    {
      "name": "gray",
      "shape": [
        1,
        144,
        256,
        1
      ],
      "dtype": "int8",
      "scale": 0.0078125,
      "zero_point": 0
    }
  ],
  "outputs": [
    {
      "name": "Identity",
      "shape": [
        1,
        18,
        32,
        1
      ],
      "dtype": "int8",
      "scale": 0.05208609253168106,
      "zero_point": 65
    },
    {
      "name": "Identity_1",
      "shape": [
        1,
        18,
        32,
        4
      ],
      "dtype": "int8",
      "scale": 0.007579573430120945,
      "zero_point": -69
    }
  ],
  "ops": [
    "CONV_2D",
    "CONV_2D",
    "CONV_2D",
    "CONV_2D",
    "CONV_2D",
    "CONV_2D",
    "CONV_2D"
  ],
  "calibration_count": 480,
  "calibration_split": "train",
  "full_int8": true,
  "target_micro_invoke_verified": false,
  "hardware_verified": false,
  "

In [5]:
outputs=predict_tflite(path,(vx-128)/128)
score=next(x for x in outputs if x.shape[-1]==1);boxes=next(x for x in outputs if x.shape[-1]==4)
q=np.floor(np.concatenate([score,boxes],axis=-1)*256+.5).astype(np.int32)
f=np.floor(model.predict((vx-128)/128,batch_size=48,verbose=0)*256+.5).astype(np.int32)
thresholds=[-768,-512,-384,-256,-128,0,128,256,512,768]
results=[stats(q,validation,t) for t in thresholds]
eligible=[r for r in results if r['precision_iou50']>=.85 and r['scene_recall_iou50']>=.50]
report=dict(version='v1.3',architecture='Five standard convolutions; expanded5x5 initialized from trained dilated3x3; relative box loss; separate INT8 heads',epochs=12,model_sha256=digest(out/'selected.keras'),tflite_sha256=digest(path),validation=results,float_validation=[stats(f,validation,t) for t in thresholds],deployment_gate_pass=bool(eligible),selected=max(eligible,key=lambda r:r['scene_recall_iou50']) if eligible else None,test_evaluated=False,hardware_verified=False)
(out/'validation_report.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))


{
  "version": "v1.3",
  "architecture": "Five standard convolutions; expanded5x5 initialized from trained dilated3x3; relative box loss; separate INT8 heads",
  "epochs": 12,
  "model_sha256": "73515abcbf98f765297a79468c90c72807be9a10560193408d23f6eaf2ba9a29",
  "tflite_sha256": "27e969f1614a9875d6d95ebfb4664422ba8de80f6c1f44afec0ea33938d21019",
  "validation": [
    {
      "threshold_q8": -768,
      "images": 628,
      "positive_images": 432,
      "background_images": 196,
      "accepted": 601,
      "localized_iou50": 201,
      "precision_iou50": 0.33444259567387685,
      "scene_recall_iou50": 0.4652777777777778,
      "f1": 0.38915779283639884,
      "background_false_accept": 169,
      "mean_best_iou": 0.2716767010594676
    },
    {
      "threshold_q8": -512,
      "images": 628,
      "positive_images": 432,
      "background_images": 196,
      "accepted": 549,
      "localized_iou50": 201,
      "precision_iou50": 0.366120218579235,
      "scene_recall_iou50": 0.46527

## 下一步
门槛未通过：继续训练/诊断，保留实验TFLite用于算子兼容性检查，不发布部署结果。门槛通过：冻结阈值后再运行未使用的测试集和全流程定位+分类评估，再做目标TFLite Micro及硬件验证。